# tse_tick — Basic Usage

A hands-on tutorial for **extracting Nikkei NEEDS tick data** with `tse_tick`. By the end you'll be
able to point the package at your data, pull exactly the ticker / date / time slice you want,
explore it, and export it.

> **Note:** `tse_tick` is built on **[Polars](https://pola.rs)** — every loader returns a *Polars*
> DataFrame (not pandas). The cells below use Polars idioms; Section 11 shows how to convert to
> pandas if you prefer.

**What you'll do**
1. Install the package
2. Get your data onto local disk
3. Configure paths (one cell)
4. Load a file
5. **Extract the slice you want** ← the main event
6. Tour the four data types, switch column language, explore, and export

## 1. Installation

```bash
pip install tse-tick            # core (polars, pyarrow)
pip install "tse-tick[query]"   # + DuckDB, for querying a pre-built Parquet store
```

Working from a clone of the repo instead? Install it in editable mode: `pip install -e ".[dev]"`.

## 2. Getting your data

`tse_tick` does **not** ship the data — it reads **Nikkei NEEDS** tick files, which require an
institutional subscription. You point the package at the raw ZIP files on your local disk.

**If your data is shared via Google Drive** (as it is here), mirror it to local disk first. The
repo's [`rclone_guide.md`](../../rclone_guide.md) walks through this end to end; the short version is:

```bash
rclone copy "REMOTE:<dataset>/<year>/<product>/" "TSE_DATA/<year>/<product>/" --drive-shared-with-me --progress
```

**What the files look like.** NEEDS is delivered as daily ZIPs named `H<TYPE>.<date>.<part>.zip`,
one or more *parts* per day:

| File prefix | `data_type`        | Output cols | What it is |
|-------------|--------------------|:-----------:|------------|
| `HTICST120` | `individual_stock` | 95 | Per-stock tick executions + 10-level order book (intraday) |
| `HTICSS110` | `stock_summary`    | 82 | Per-stock daily summary (OHLC, VWAP, volumes) |
| `HTICIT110` | `indices`          | 10 | Index tick updates — Nikkei 225, TOPIX, … (intraday) |
| `HTICIS110` | `indices_summary`  | 17 | Per-index daily summary |

Supported years: **2016–2025** (2016 index files use a legacy fixed-width format that the library
detects and parses automatically — no action needed).

**Folder layout.** Deliveries are usually organised as `{year}/{yearmonth}/…`:

```
TSE_DATA/
  2024/
    202402/
      HTICST120.20240201.1.zip
      HTICST120.20240201.2.zip   # a single day can span several numbered parts
      ...
```

You don't have to match this exactly: **point the package at any parent folder** and it locates the
files by *type + date*, regardless of folder names or depth. One consequence — a single numbered ZIP
is only *part* of a day, so filtering one lone `….N.zip` by ticker can return 0 rows. For complete
coverage point at the **day's folder (or a parent)**, which `read_ticks` handles for you.

## 3. Import and check your install

Printing the version confirms the install. `get_info()` is a quick orientation banner — the data
types, the supported year range, and the two ways to get data out.

In [ ]:
import tse_tick
import polars as pl
from pathlib import Path

print("tse_tick:", tse_tick.__version__)
print("polars:  ", pl.__version__)

# tse_tick returns *Polars* DataFrames. These just make the previews roomier.
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(15)


def configured(path: str) -> bool:
    """True if `path` exists; else print a friendly 'edit CONFIG' note.

    Lets you run the whole notebook top-to-bottom before setting your paths:
    cells that need data print a hint instead of raising.
    """
    if path and Path(path).exists():
        return True
    print(f"[!] Not found: {path!r}\n"
          f"    -> Edit the CONFIG cell (Section 4) to point at your data, then re-run this cell.")
    return False

In [ ]:
# Data types, year range, and the two access patterns. get_info() RETURNS the
# banner (it doesn't print), so wrap it in print().
print(tse_tick.get_info())

## 4. Configure — *edit this cell*

Everything below reads from the variables set here, so this is the **only cell you need to edit**.
Point `STOCK_FILE` / `DATA_ROOT` at your own data (see Section 2) and pick the ticker, date, and time
window you want to extract.

Until you set real paths, data cells print a small *"Not found — edit CONFIG"* note instead of
failing, so you can safely run the whole notebook first to read along.

In [ ]:
# ============================================================================
#  CONFIG  --  edit me, then run the notebook top to bottom
# ============================================================================

# A folder containing your NEEDS ZIPs. Can be a single .zip, a flat folder, or
# any parent of the {year}/{yearmonth}/ tree -- files are found by type + date.
DATA_ROOT = r"path/to/TSE_DATA"

# A single individual-stock (TICST120) ZIP for the "load one file" demos.
STOCK_FILE = r"path/to/HTICST120.20240201.1.zip"

# What to EXTRACT in Section 6:
TICKER     = "7203"               # Toyota. Stock codes are strings -- keep the quotes.
DATE       = "20240201-20240205"  # a day "20240201", month "202402", year "2024", or a range
START_TIME = "09:00:00"           # intraday window (tick data only)
END_TIME   = "11:30:00"

# Example files for the other three data types (Section 7):
SUMMARY_FILE       = r"path/to/HTICSS110.202402.zip"  # stock_summary  (monthly)
INDEX_FILE         = r"path/to/HTICIT110.202402.zip"  # indices
INDEX_SUMMARY_FILE = r"path/to/HTICIS110.202402.zip"  # indices_summary

## 5. Load a data file (`create_df`)

`create_df()` reads one ZIP (or a whole folder of ZIPs) into a Polars DataFrame. It **auto-detects**
the data type and year from the filename, names the columns, and cleans the types.

> **Heads-up on memory:** a full individual-stock day can be millions of rows. `create_df` reads it
> all in one shot (and raises a catchable `OneShotMemoryError` if it's too big). For a quick look
> pass `rows=`; for large or repeated work, ingest to a Parquet store and use `query_ticks`
> (Section 6.4).

In [ ]:
df = None
if configured(STOCK_FILE):
    df = tse_tick.create_df(STOCK_FILE, language="en")   # -> Polars DataFrame
    print(f"Loaded {df.height:,} rows x {df.width} columns")
df  # rich preview (shows nothing until STOCK_FILE is set)

### 5.1 Sample a few rows

Pass `rows=N` to read only the first N rows — handy while iterating.

In [ ]:
df_sample = None
if configured(STOCK_FILE):
    df_sample = tse_tick.create_df(STOCK_FILE, language="en", rows=1000)
    print(f"Sampled {df_sample.height:,} rows")
df_sample

### 5.2 Skip auto-detection

If a file lives in a folder whose name carries no year/type, turn auto-detection off and pass
`data_type` and `year` yourself.

In [ ]:
df_explicit = None
if configured(STOCK_FILE):
    df_explicit = tse_tick.create_df(
        STOCK_FILE,
        auto_detect=False,
        data_type="individual_stock",
        year=2024,
        rows=5,
    )
    print(f"OK -- {df_explicit.height} rows")
df_explicit

## 6. Extract the slice you want

This is the part you'll use most. Two ways to get a **filtered** DataFrame:

- **One-shot** (`read_ticks`) — read straight from the raw ZIPs to a ticker/date/time-filtered frame,
  no store to build first. Best for one or a few tickers over a bounded window. *(Used below.)*
- **Two-stage** (`ingest_* → query_ticks`) — convert the ZIPs to a Parquet store once, then query it
  repeatedly (far faster for repeated/large work). See Section 6.4.

### 6.1 Filter by ticker while loading a file

`create_df(..., ticker_filter={"7203"})` keeps only the codes you ask for (a **set of string**
codes). Remember a single numbered ZIP is only *part* of a day, so a lone part may not contain your
ticker — point at the day's folder (Section 6.2) for complete coverage.

In [ ]:
df_one = None
if configured(STOCK_FILE):
    df_one = tse_tick.create_df(STOCK_FILE, language="en", ticker_filter={TICKER})
    print(f"{TICKER}: {df_one.height:,} rows in this ZIP part")
df_one

### 6.2 Pull a ticker over a date + time window (`read_ticks`)

`read_ticks` discovers the right ZIPs under `DATA_ROOT`, reads **every part** of each day (so the
result is complete), and filters by ticker, date, and intraday time window. `date` accepts a single
day, a month, a year, or a range. A read that matches nothing returns an *empty but fully-typed*
frame and warns (`NoDataWarning`).

In [ ]:
slice_df = None
if configured(DATA_ROOT):
    slice_df = tse_tick.read_ticks(
        DATA_ROOT,
        data_type="individual_stock",
        ticker_filter={TICKER},
        date=DATE,
        start_time=START_TIME,
        end_time=END_TIME,
    )
    print(f"Extracted {slice_df.height:,} rows for {TICKER} "
          f"over {DATE} ({START_TIME}-{END_TIME})")
slice_df

### 6.3 Keep only the columns you need

Pass `columns=[...]` to project just those fields — smaller, faster results.

In [ ]:
cols = ["Data Date", "Execution Time", "Stock Code", "Execution Price", "Volume"]
slice_small = None
if configured(DATA_ROOT):
    slice_small = tse_tick.read_ticks(
        DATA_ROOT,
        data_type="individual_stock",
        ticker_filter={TICKER},
        date=DATE,
        columns=cols,
    )
    print(f"{slice_small.height:,} rows x {slice_small.width} columns")
slice_small

### 6.4 The other path: ingest once, query many times

For repeated or large-scale work, build a Parquet store once and query it:

```python
tse_tick.ingest_period(DATA_ROOT, "STORE", period="202402", data_type="individual_stock")
df = tse_tick.query_ticks("STORE", ticker=7203, date="20240201",
                          start_time="09:00:00", end_time="11:30:00")
```

`query_ticks` needs the `[query]` extra (DuckDB). The companion notebook **`02_evaluation.ipynb`**
exercises both paths against real data; the README documents the full CLI (`tse-tick ingest` /
`tse-tick export`).

## 7. The four data types

The same `create_df` / `read_ticks` calls work for all four NEEDS types — just point at the matching
files. Each cell skips itself with a note if that file isn't configured.

In [ ]:
# 1) individual_stock (TICST120) -- per-stock ticks + order book
if configured(STOCK_FILE):
    d = tse_tick.create_df(STOCK_FILE, language="en", rows=1000)
    print("individual_stock:", d.shape)
    print(d.select(["Stock Code", "Execution Time", "Execution Price", "Volume"]).head())

In [ ]:
# 2) stock_summary (TICSS110) -- per-stock daily aggregates (monthly file)
if configured(SUMMARY_FILE):
    d = tse_tick.create_df(SUMMARY_FILE, language="en")
    print("stock_summary:", d.shape)
    print(d.select(["Stock Code", "Daily VWAP", "AM Total Volume", "PM Total Volume"]).head())

In [ ]:
# 3) indices (TICIT110) -- index tick updates (Nikkei 225, TOPIX, ...)
if configured(INDEX_FILE):
    d = tse_tick.create_df(INDEX_FILE, language="en")
    print("indices:", d.shape)
    print("columns:", d.columns)
    print(d.head())

In [ ]:
# 4) indices_summary (TICIS110) -- per-index daily summary
if configured(INDEX_SUMMARY_FILE):
    d = tse_tick.create_df(INDEX_SUMMARY_FILE, language="en")
    print("indices_summary:", d.shape)
    print(d.head())

## 8. Language support

Every loader takes `language="en"` (default) or `language="jp"`. `get_japanese_column_mapping()`
returns the full English→Japanese dictionary.

In [ ]:
if configured(STOCK_FILE):
    en = tse_tick.create_df(STOCK_FILE, language="en", rows=100)
    jp = tse_tick.create_df(STOCK_FILE, language="jp", rows=100)
    print("English  (first 10):", en.columns[:10])
    print()
    print("Japanese (first 10):", jp.columns[:10])

In [ ]:
mapping = tse_tick.get_japanese_column_mapping()
print(f"{len(mapping)} column-name mappings (English -> Japanese). First 20:\n")
for en_name, jp_name in list(mapping.items())[:20]:
    print(f"  {en_name:32} {jp_name}")

## 9. Explore the data

Standard Polars exploration on the frame `df` you loaded in Section 5.

### 9.1 Types and summary statistics

In [ ]:
if df is not None:
    for name in ["Stock Code", "Execution Time", "Execution Price", "Volume"]:
        if name in df.columns:
            print(f"{name:16} -> {df[name].dtype}")
    print()
    print(df.describe())

### 9.2 Unique values

Polars uses `.n_unique()` (not pandas' `.nunique()`).

In [ ]:
if df is not None and "Stock Code" in df.columns:
    print("Unique stock codes:", df["Stock Code"].n_unique())
    print("\nMost frequent codes:")
    print(df["Stock Code"].value_counts(sort=True).head(10))

if df is not None and "Session" in df.columns:
    print("\nSessions present:", df["Session"].unique().to_list())

### 9.3 Missing values

Polars counts nulls with `df.null_count()` (a one-row frame), not pandas' `df.isnull().sum()`.

In [ ]:
if df is not None:
    counts = dict(zip(df.columns, df.null_count().row(0)))
    missing = {c: n for c, n in counts.items() if n > 0}
    if missing:
        print("Columns with nulls (count, % of rows):")
        for c, n in sorted(missing.items(), key=lambda kv: kv[1], reverse=True):
            print(f"  {c:28} {n:>10,}  ({n / df.height:.1%})")
    else:
        print("No missing values in this sample.")

### 9.4 Filter one stock

`Stock Code` is a **string** column, so compare to `"7203"` (not the integer `7203`).

In [ ]:
if df is not None and "Stock Code" in df.columns:
    one = df.filter(pl.col("Stock Code") == TICKER)
    print(f"{TICKER}: {one.height:,} rows")
    print(one.head())

## 10. Export to CSV

`export_to_csv` reads a file and writes a CSV in one step (auto-named or to `output_path`, in either
language). You can also write any extracted Polars frame directly with `.write_csv(...)`.

In [ ]:
if configured(STOCK_FILE):
    out1 = tse_tick.export_to_csv(STOCK_FILE, language="en", rows=1000)      # auto-named
    print("Wrote:", out1)

    out2 = tse_tick.export_to_csv(STOCK_FILE, output_path="stock_sample.csv",  # custom name
                                  language="en", rows=1000)
    print("Wrote:", out2)

    out3 = tse_tick.export_to_csv(STOCK_FILE, output_path="kabushiki_data.csv",  # Japanese columns
                                  language="jp", rows=1000)
    print("Wrote:", out3)

In [ ]:
# Write any extracted frame directly:
if slice_df is not None and slice_df.height:
    out = f"{TICKER}_{DATE}.csv"
    slice_df.write_csv(out)
    print(f"Wrote {out}  ({slice_df.height:,} rows)")

## 11. (Optional) Convert to pandas

`tse_tick` returns Polars frames. If your downstream code expects pandas, convert with `.to_pandas()`
(needs `pip install pandas`; the Arrow bridge is already a core dependency).

In [ ]:
if df is not None:
    try:
        pdf = df.to_pandas()
        print(type(pdf).__module__, "->", type(pdf).__name__, pdf.shape)
        print(pdf.head())
    except ImportError:
        print("pandas not installed -- `pip install pandas` to use df.to_pandas().")

## Summary & next steps

You loaded NEEDS data, **extracted a ticker over a date/time window** with `read_ticks`, toured the
four data types, switched column languages, explored with Polars, and exported to CSV.

**Where to go next**
- **`02_evaluation.ipynb`** — both access paths (one-shot vs ingest→query) and order-book features,
  validated against real data
- **`README.md`** — the full Python API, the `tse-tick` CLI, and the Parquet store layout
- **`rclone_guide.md`** — mirroring your Google Drive data to local disk
- Repository: <https://github.com/tse-tick/tse_tick>